# Prithvi-WxC Inference

This notebook run inference using the finetuned Prithvi-WxC model.

In [1]:
%load_ext autoreload
%autoreload 2
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import xarray as xr

The environment variable below should point to the folder containing the ``musigma_surface.nc`` and ``musigma_vertical.nc`` files.

In [2]:
%env PRITHVI_DATA_PATH=/mnt/ssd-data2/TEMP/hf_cache/climatology/

env: PRITHVI_DATA_PATH=/mnt/ssd-data2/TEMP/hf_cache/climatology/


In [3]:
!pip install peft

## Load the model

In [84]:
from pytorch_retrieve.architectures import load_model

mdl = load_model("/mnt/ssd-data1/prithvi_tc/prithvi_tc_v0.1.pt")

## Merged Data

The functions below provide functionality to load merged U10, anc V10 fields.

In [33]:
from datetime import datetime

def get_date(path: Path) -> datetime:
    """ Extract timestamp from merged MERRA file."""
    date = path.name.split('_')[-2]
    return datetime.strptime(date, "%Y%m%d%H")
    
merged_files = {}
files = sorted(list(Path("/mnt/ssd-data2/TEMP/merra_blended_nc/").glob("**/*.nc")))
for path in files:
    try:
        date = get_date(path)
        merged_files[date] = path
    except ValueError as exc:
        raise exc
        
print(f"Found {len(merged_files)} merged files.")

def load_merged_analysis(time: np.datetime64) -> xr.Dataset:
    """
    Load merged data for given initialization time.
    """
    time = time.astype("datetime64[s]").item()
    merged_file = merged_files[time]
    with xr.open_dataset(merged_file) as data:
        data = data[{"lat": slice(0, -1), "time": 0}][["mu10", "mv10", "vmax_nan"]].compute()
    return data

Found 75976 merged files.


In [34]:
load_merged_analysis(np.datetime64("2020-01-01"))

<xarray.Dataset> Size: 2MB
Dimensions:   (lat: 360, lon: 576)
Coordinates:
  * lat       (lat) float64 3kB -90.0 -89.5 -89.0 -88.5 ... 88.0 88.5 89.0 89.5
  * lon       (lon) float64 5kB -180.0 -179.4 -178.8 ... 178.1 178.8 179.4
    time      datetime64[ns] 8B 2020-01-01
Data variables:
    mu10      (lat, lon) float32 829kB nan nan nan nan nan ... nan nan nan nan
    mv10      (lat, lon) float32 829kB nan nan nan nan nan ... nan nan nan nan
    vmax_nan  (lat, lon) float32 829kB nan nan nan nan nan ... nan nan nan nan
Attributes: (12/30)
    History:                           Original file generated: Thu Sep 26 14...
    Comment:                           GMAO filename: d5124_m2_jan10.inst1_2d...
    Filename:                          MERRA2_400.inst1_2d_asm_Nx.20190915.nc4
    Conventions:                       CF-1
    Institution:                       NASA Global Modeling and Assimilation ...
    References:                        http://gmao.gsfc.nasa.gov
    ...                                ...
    Contact:                           http://gmao.gsfc.nasa.gov
    identifier_product_doi:            10.5067/3Z173KIE2TPD
    RangeBeginningDate:                2019-09-15
    RangeBeginningTime:                00:00:00.000000
    RangeEndingDate:                   2019-09-15
    RangeEndingTime:                   23:00:00.000000

In [35]:
import torch
from typing import Dict

def post_process_results(
    inpt: Dict[str, torch.Tensor],
    results: Dict[str, torch.Tensor],
    init_times: np.ndarray,
    valid_times: np.ndarray,
) -> xr.Dataset:
    """
    Extracts surface winds, surface pressure and 850 winds from forecast results.

    Args:
        inpt: The batch containing the input data.
        results: A dictionary containing the forecast results.

    Return:
        An xarray.Dataset containing the results to be stored.
    """
    static = inpt["static"]
    if static.dim() == 5:
        lats = np.rad2deg(inpt["static"][0, 0, 0, :, 0].float().cpu().numpy())
        lons = np.rad2deg(inpt["static"][0, 0, 1, 0, :].float().cpu().numpy())
    else:
        lats = np.rad2deg(inpt["static"][0, 0, :, 0].float().cpu().numpy())
        lons = np.rad2deg(inpt["static"][0, 1, 0, :].float().cpu().numpy())

    var_inds = [9, 17, 18, -18, -4]

    # Stack output steps into single array
    pred = np.stack([step[:, [9, 17, 18, -18, -4]].float().cpu().numpy() for step in results["y"]], axis=1)
    # Add analysis to forecast results
    analysis = inpt["x"][:, 1:, var_inds].float().cpu().numpy()
    pred = np.concatenate((analysis, pred), 1)

    u10 = np.stack([step[:, 0].float().cpu().numpy() for step in results["u10"]], axis=1)
    v10 = np.stack([step[:, 0].float().cpu().numpy() for step in results["v10"]], axis=1)
    vmax = np.stack([step[:, 0].float().cpu().numpy() for step in results["vmax"]], axis=1)

    merged_analysis = xr.concat([load_merged_analysis(time) for time in init_times], dim="batch").transpose("batch", ...)
    u10 = np.concatenate((merged_analysis.mu10.data[None], u10), 1)
    v10 = np.concatenate((merged_analysis.mv10.data[None], v10), 1)
    vmax = np.concatenate((merged_analysis.vmax_nan.data[None], vmax), 1)
    
    valid_times = np.concatenate([init_times.reshape((init_times.shape[0], 1)), valid_times], 1)
    
    dataset = xr.Dataset({
        "initialization_time": (("batch",), init_times),
        "valid_time": (("batch", "step",), valid_times),
        "latitude": (("latitude",), lats),
        "longitude": (("longitude",), lons)
    })
    
    slp = pred[:, :, 0]
    u850 = pred[:, :, 3]
    v850 = pred[:, :, 4]

    dataset["slp"] = (("batch", "step", "latitude", "longitude"), slp)
    dataset["u10"] = (("batch", "step", "latitude", "longitude"), u10)
    dataset["v10"] = (("batch", "step", "latitude", "longitude"), v10)
    dataset["u850"] = (("batch", "step", "latitude", "longitude"), u850)
    dataset["v850"] = (("batch", "step", "latitude", "longitude"), v850)
    dataset["vmax"] = (("batch", "step", "latitude", "longitude"), vmax)

    for var in ["slp", "u10", "v10", "u850", "v850", "vmax"]:
        dataset[var].encoding = {"dtype": "float32", "zlib": True}

    return dataset


### Data Loader

The data loader loads the input data for the forecasts.

In [56]:
from pathlib import Path
from prithvi_precip.forecast.data_loaders import AutoregressiveForecastLoader
from torch.utils.data import DataLoader

input_data_path = Path("/mnt/ssd-data1/prithvi_tc/input_data/")

# Adapt this to run forecast for longer time range.
init_times = np.arange(
    np.datetime64("2021-08-27T18:00:00"),
    np.datetime64("2021-08-28T00:00:00"),
    np.timedelta64(6, "h")
)

data_loader = AutoregressiveForecastLoader(
    input_data_path,
    init_times=init_times,
    n_steps=20, # Number of 6-hour forecast steps.
    input_time=6, # Input data time step
    center_meridionally=False,
    full_climatology=True,
)
data_loader = DataLoader(data_loader, batch_size=None, num_workers=1, collate_fn=lambda x: x)
print(f"Found input data for {len(data_loader)} forecasts.")

Found input data for 1 forecasts.


## Run the inference

In [85]:
from prithvi_precip.forecast.runners import run_autoregressive_forecast

output_path = Path("/mnt/ssd-data1/prithvi_tc/results_v0.1")
output_path.mkdir(exist_ok=True)

device = "cuda:1"

run_autoregressive_forecast(
    mdl,
    data_loader,
    output_path,
    post_process_fn=post_process_results,
    device=device,
    dtype=torch.float32,
)

100%|████████████████████████████████████████████████████████████████████| 1/1 [05:14<00:00, 314.32s/it]
